This is a notebook to visualize different properties of the dataset as we go about curating. Here, we're extracting specific annotations from the tsv we downloaded from [UniProt](https://www.uniprot.org/) since this gives detailed residue-level information that we can try to predict. 

In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import re
import json
from typing import List
import polars as pl
import plotly.express as px

from project.utils.strs import SEED, data_dir
from project.utils.splitting import hierarchical_clustered_split

Helper functions

In [ ]:
def annotate(features_str: str, feature_length: int, blank: str = '-', add_note=True) -> List[str]:
    """
    Parses a features string and updates a list with feature names or /note content,
    supporting both range (START..END) and point (POSITION) annotations.
    """
    # Initialize the feature list
    feature_list = [blank for _ in range(feature_length)]

    if features_str is None:
        return feature_list
    
    # Pattern:
    # 1. (\w+)\s+             - Group 1: Feature Code (e.g., 'LIPID')
    # 2. (\d+)                - Group 2: Start Position (or single position)
    # 3. (?:..(\d+))?         - Group 3 (Optional): Matches '..' followed by the End Position.
    #                          - If a range (..END) is not present, this group will be None.
    # 4. ;(?:.*?/note="([^"]+?)")? - Group 4 (Optional): Captures the /note content.
    feature_pattern = r'(\w+)\s+(\d+)(?:..(\d+))?;(?:.*?/note="([^"]+?)")?'
    
    # Find all matches (re.DOTALL allows '.' to match newlines if they existed)
    matches = re.finditer(feature_pattern, features_str, flags=re.DOTALL)
    
    for match in matches:
        # Groups: (Code, Start, End, Note)
        feature_name, start_str, end_str, note_content = match.groups()
        
        # Determine the value to place in the list
        if add_note and (note_content is not None) and not ('(' in note_content) and not (')' in note_content):
            # If there's a semicolon, strip it out
            if ';' in note_content:
                note_content = ''.join(note_content.split(';')[0])
            
            fill_value = note_content
        else:
            fill_value = feature_name
            
        # Handle coordinates
        start = int(start_str)
        # If the '..END' part was matched, end_str will contain the number.
        # If it was a point annotation (e.g., 'LIPID 667;'), end_str will be None.
        end = int(end_str) if end_str else start 
        
        # Fill the list positions (adjusting for 0-based indexing)
        # The range is from start - 1 up to (but not including) end. 
        # For a point (start=end), this covers exactly one position.
        for i in range(start - 1, end):
            if i < len(feature_list):
                feature_list[i] = fill_value.replace(' ', '_')
                
    return feature_list


def combine_many_feature_lists(feature_lists: List[List[str]], blank: str = '-') -> List[str]:
    """
    Combines an arbitrary number of feature lists element-wise.
    """
    if not feature_lists:
        return []

    # All lists are assumed to be the same length (Polars enforces this structure)
    
    # Use zip to iterate through all lists simultaneously, getting a tuple of values per position
    # E.g., for position 'i', vals_at_pos = (list1[i], list2[i], list3[i], ...)
    combined_list = []
    
    # Transpose the lists using zip and convert to a list of tuples
    transposed_lists = list(zip(*feature_lists))
    
    for vals_at_pos in transposed_lists:
        # Find all values at the current position that are NOT the blank character
        labels = [val for val in vals_at_pos if val != blank]
        
        if not labels:
            # Case 1: All values are blank
            combined_list.append(blank)
        else:
            # Case 2/3: One or more labels are present
            # Use a set to get unique labels, then join them
            unique_labels = sorted(list(set(labels)))
            # if len(unique_labels) > 1:
            #     print(f"label overlap! {unique_labels}")
            combined_list.append(",".join(unique_labels))
            
    return combined_list

In [ ]:
subset_dir = data_dir / 'processed_subsets'
ann_dir = subset_dir / 'uniprot_ann'
ann_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Thresholds - must be separate clusters in this mmseqs category to be put into test set
val_threshold_value = 50 # % identity minimum
test_threshold_value = 20 # 

val_threshold = f'mmseqs_0.{val_threshold_value}'
test_threshold = f'mmseqs_0.{test_threshold_value}'

### Reading in the dataset

In [ ]:
# Read annotated dataset
annotated_data = 'uniprotkb_AND_model_organism_9606_2025_09_09_annotated.parquet.gz'
df = pl.read_parquet(data_dir / annotated_data)
# We always want these columns
base_cols = ['id', 'sequence', 'Length']
mmseqs_cols = [c for c in df.columns if ('mmseqs') in c]

### Processing and Splitting

In [ ]:
num_splits = 3
datasets = {}

### Helper functions
# We'll combine results from these columns since they should largely be non-overlapping (or if they do overlap, it may be useful information)
cols_to_combine = {
    'secondary_structure': ['Helix', 'Beta strand', 'Turn'],
    'post_translational_modification': ['Modified residue', 'Lipidation', 'Disulfide bond', 'Glycosylation'],
    'glycosylation': ['Glycosylation'],
    'phosphorylation': ['Modified residue'],
    'lipidation': ['Lipidation'],
    'membrane_pass': ['Transmembrane', 'Intramembrane'], 
    'topology': ['Topological domain'],
    'peptide': ['Propeptide', 'Signal peptide', 'Transit peptide'],
    "functional_sites": ["Active site", "Binding site", 'DNA binding'],
    "domains": ["Domain [FT]"],
    'regions': ["Region"],
    'structures': ["Repeat","Coiled coil", "Zinc finger"],
}

placeholder = '-' # What to fill empty residues with
mappings = {} # To keep track of mappings between classes and the numbers we assign them

for grouping, columns in cols_to_combine.items():
    print(grouping)
    # Clone a subset df with only chosen columns
    df_subset = df.clone().select(base_cols + mmseqs_cols + columns)
    residues_col_names = []
    # Add new columns, for each per-residue annotation we want
    for target_col in columns:
        # New column name
        residues_col = f"{target_col.replace(' ', '_').lower()}_per_residue"
        residues_col_names.append(residues_col)
        
        # Use the length to make a list of the same length as the protein - we will fill in this list with annotations
        # Inefficient way to do this put polars didn't like applying a list of expressions to with_columns
        df_subset = df_subset.with_columns(
            pl.struct([
                pl.col(target_col), 
                pl.col('Length')
            ]) # Use pl.struct to bundle the feature string and the length.
            .map_elements(lambda x: annotate(
                features_str=x[target_col], 
                feature_length=x['Length'],
                blank = placeholder,
                add_note=False,
            ), return_dtype=pl.List(pl.String))
            .alias(residues_col)
        )
        # For these, we want to extract the note
        if (target_col in ['Topological domain', 'Glycosylation', 'Modified residue', 'Lipidation']) and (grouping in ['glycosylation', 'phosphorylation', 'lipidation', 'topology']):
            df_subset = df_subset.with_columns(
            pl.struct([
                pl.col(target_col), 
                pl.col('Length')
            ]) # Use pl.struct to bundle the feature string and the length.
            .map_elements(lambda x: annotate(
                features_str=x[target_col], 
                feature_length=x['Length'],
                blank = placeholder,
                add_note=True,
            ), return_dtype=pl.List(pl.String))
            .alias(residues_col)
        )

    # Combine results from residues cols together
    struct_exprs = [pl.col(name) for name in residues_col_names] # convert to struct for with_columns to use
    df_subset = df_subset.with_columns(
        pl.struct(struct_exprs)
        .map_elements(
            lambda x: combine_many_feature_lists(
                feature_lists=list(x.values()), 
                blank=placeholder,
            ),
            return_dtype=pl.List(pl.String)
        )
        .alias(grouping)
    )

    def replace_all_but_phospho(list_to_replace):
        to_return = []
        for i in list_to_replace:
            if 'phospho' in i.lower():
                to_return.append(i)
            else:
                to_return.append('-')
        return to_return

    if grouping == 'phosphorylation':
        # We only want phospho labels - turn everything else into a '-'
        phospho_replacement = {'-'}
        # make a column indicating which rows have which classes
        df_subset = df_subset.with_columns(
            pl.col(grouping).map_elements(lambda x: replace_all_but_phospho(x)))

    # Assign class numbers by finding unique categories, assigning a number, and adding a column with those numbers
    classes = sorted(df_subset[grouping].explode().unique())
    class_mapping = {c: i for i,c in enumerate(classes)}
    # Exchange class labels for corresponding ints, and make a column indicating which rows have which classes
    df_subset = df_subset.with_columns(
        pl.col(grouping).list.eval(pl.element().replace_strict(class_mapping)).alias('targets'),    # replace classes amino-acid-wise with target classes
    ).with_columns(
        pl.col('targets').list.unique().alias('classes'), # for convenience, track which classes are there
    ).with_columns(
        pl.col('classes').list.eval(pl.element().replace_strict({v:k for k,v in class_mapping.items()})).alias('unique_class_names')
    )

    # Drop most rows where target columns are zero ie. where we only have 'class' = [0]
    df_positive = df_subset.filter(pl.col('classes').list.sum() !=0 )
    df_negative = df_subset.filter(pl.col('classes').list.sum() ==0 ) # Get all the negative samples
    df_negative = df_negative.sample(min(len(df_negative), int(len(df_positive)/3)), seed=SEED) # Either get all the rest of the negative samples, or a proportion of the positive samples (2:1 positive / negative)

    # combine the positives and negatives together
    df_subset = pl.concat([df_positive, df_negative])

    for i in range(num_splits):
        if i !=0:
            appendix = '_split' + str(i)
        else:
            appendix = ''

        # Apply split by mmseqs threshold
        df_subset = pl.concat(hierarchical_clustered_split(df_subset, 
                    val_threshold = val_threshold,
                    test_threshold = test_threshold,
                    val_ratio = 0.15,
                    test_ratio = 0.15,
                    seed = SEED+i))

        # # Write out as parquet
        df_subset.write_parquet(ann_dir / f"h_sapiens_proteome_uniprot_{grouping}_clustersplit_{test_threshold_value}_{val_threshold_value}{appendix}.parquet.gz")

        #make a smaller version of the df
        print('before subsetting')
        print(df_subset['split'].value_counts(normalize=True))
        # Make a dataset that's a subset for the checkpoints experiments - it has a max context length of 512 tokens.
        df_512 = df_subset.filter(pl.col('sequence').str.len_chars() <= 512)
        print('after subsetting')
        print(df_512['split'].value_counts(normalize=True))
        df_512.write_parquet(ann_dir / f"h_sapiens_proteome_uniprot_{grouping}_clustersplit_{test_threshold_value}_{val_threshold_value}_512_cutoff{appendix}.parquet.gz")

    # Keep track of class:int for this group
    mappings[grouping] = class_mapping

    datasets[grouping] = df_subset

    del df_subset

# Export the mappings as a json

with open(ann_dir / f"h_sapiens_proteome_uniprot_grouping_classes.json", 'w') as f:
    json.dump(mappings, f)

In [ ]:
def dataset_stats(df, dataset_name):
    """
    For uniprot-parsed datasets or biomap amino-acid-level datasets, get custom high-level stats on how many proteins contain a certain class, and what percent of the protein is dedicated to that class.
    """

    from pathlib import Path
    import json
    fig_out= Path('/home/mila/s/shawn.whitfield/scratch/results/dataset_characterization')
    
    if df['targets'].dtype.is_integer():
        def wrap(int):
            return [int]
        df = df.with_columns(
            pl.col('targets').map_elements(lambda x: wrap(x), return_dtype=pl.List(pl.Int64))
        )

    if 'classes' not in df.columns:
        df = df.with_columns(
            pl.col('targets').list.unique().alias('classes')
        )
    if 'Length' not in df.columns:
        df = df.with_columns(
            pl.col('targets').list.len().alias('Length')
        )


    class_percents = {}

    classes = list(df['unique_class_names'].explode().unique())

    print(classes)

    # Percent of each class, per protein
    for class_num in classes:
        df = df.with_columns(
            (pl.col(dataset_name.lower()).list.count_matches(class_num) / pl.col('Length')).alias(f'percent_{class_num}')
        )
        class_percents[class_num] = (df[f'percent_{class_num}'] > 0).sum() / len(df)

    fig = px.box(df.select([f'percent_{n}' for n in classes] + ['split']), color='split')
    fig.update_layout(title_text=f"{dataset_name} percent of [class] in each protein, {len(df)} samples", 
            xaxis_title='class', 
            yaxis_title='percent_of_protein',
            xaxis=dict(tickvals = classes, ticktext =classes),
            # width = len(columns_to_plot) * 10,
            # height=900,
            )
    fig.show()

    fig.write_html(fig_out / f"{dataset_name}_class_percent_by_protein.html")

    bar = px.bar(x=list(class_percents.keys()), y = list(class_percents.values()))
    bar.update_layout(title_text=f"{dataset_name} percent of proteins with class, {len(df)} samples", 
    yaxis_title='percent_of_proteins with class',
    xaxis=dict(tickvals = classes, ticktext =classes),
        )
    bar.show()

    bar.write_html(fig_out / f"{dataset_name}_class_percent_of_proteins.html")

    return df, class_percents

In [ ]:
for df_name, df in datasets.items():
    dataset_stats(df, dataset_name=df_name)